In [0]:
%sql
SELECT 'orders' AS tbl, COUNT(*) AS rows FROM workspace.olist_bronze.bronze_orders 
UNION ALL SELECT 'order_items', COUNT(*) FROM workspace.olist_bronze.bronze_order_items
UNION ALL SELECT 'customers', COUNT(*) FROM workspace.olist_bronze.bronze_customers
UNION ALL SELECT 'products', COUNT(*) FROM workspace.olist_bronze.bronze_products
UNION ALL SELECT 'sellers', COUNT(*) FROM workspace.olist_bronze.bronze_sellers
UNION ALL SELECT 'payments', COUNT(*) FROM workspace.olist_bronze.bronze_order_payments
UNION ALL SELECT 'reviews', COUNT(*) FROM workspace.olist_bronze.bronze_order_reviews
UNION ALL SELECT 'geolocation', COUNT(*) FROM workspace.olist_bronze.bronze_geolocation;

tbl,rows
orders,99441
order_items,112650
customers,99441
products,32951
sellers,3095
payments,103886
reviews,99224
geolocation,1000163


In [0]:
%sql
DESCRIBE workspace.olist_bronze.bronze_orders;

col_name,data_type,comment
order_id,string,null
customer_id,string,null
order_status,string,null
order_purchase_timestamp,string,null
order_approved_at,string,null
order_delivered_carrier_date,string,null
order_delivered_customer_date,string,null
order_estimated_delivery_date,string,null
_source_file,string,null
_ingested_at,timestamp,null


In [0]:
%sql
SELECT COUNT(*) AS item_rows,
COUNT(DISTINCT order_id) AS distinct_orders
FROM workspace.olist_bronze.bronze_order_items;

item_rows,distinct_orders
112650,98666


In [0]:
%sql
SELECT COUNT(*) AS total_rows,
COUNT(DISTINCT order_id) AS distinct_orders
FROM workspace.olist_bronze.bronze_orders;

total_rows,distinct_orders
99441,99441


In [0]:
%sql
SELECT COUNT(DISTINCT customer_id) AS customer_ids,
COUNT(DISTINCT customer_unique_id) AS unique_people
FROM workspace.olist_bronze.bronze_customers;

customer_ids,unique_people
99441,96096


In [0]:
%sql
SELECT customer_unique_id,
COUNT(DISTINCT customer_city) AS cities
FROM workspace.olist_bronze.bronze_customers
GROUP BY customer_unique_id
HAVING COUNT(DISTINCT customer_city) > 1
ORDER BY cities DESC
LIMIT 10;

customer_unique_id,cities
d44ccec15f5f86d14d6a2cfa67da1975,3
6e1550b8966f327f29bc06b18972dbec,2
f34cd7fd85a1f8baff886edf09567be3,2
8e84222ad36cb6ab528b745ae6e18d11,2
d55417a83a73d1702b43988e63f0731a,2
cbebce165db72a2bfba9e81a3411bdf7,2
925751a747a151a7fa97f2f686d028c3,2
508cae50c5c1e72079d266a513bca9ae,2
2b952792a20f6076f6e7a9c6a27ade02,2
7bf1c59401206e31a78523077f3a1144,2


In [0]:
%sql
SELECT c.customer_id, c.customer_city, c.customer_state,
c.customer_zip_code_prefix, o.order_purchase_timestamp
FROM workspace.olist_bronze.bronze_customers c 
JOIN workspace.olist_bronze.bronze_orders o ON o.customer_id = c.customer_id
WHERE c.customer_unique_id = 'd44ccec15f5f86d14d6a2cfa67da1975'
ORDER BY o.order_purchase_timestamp;

customer_id,customer_city,customer_state,customer_zip_code_prefix,order_purchase_timestamp
844a7e5d2737f2f6667a5242486cbcb1,sao paulo,SP,03533,2017-05-30 09:39:34
488dd91f70d8404b2aa677cdc8a0de68,navegantes,SC,88371,2017-09-13 14:55:27
105e31e3b36d904c70058578126c4d12,aracati,CE,62800,2017-11-10 13:55:00


In [0]:
%sql
SELECT COUNT(*) AS people_with_multiple_cities
FROM (
    SELECT customer_unique_id
    FROM workspace.olist_bronze.bronze_customers
    GROUP BY customer_unique_id
    HAVING COUNT(DISTINCT customer_city) > 1
);

people_with_multiple_cities
122
